# Pure PP-OCRv5 light preparation (Kaggle)

Runs **PP-OCRv5 server det + rec** on every page the light prep marks as needing OCR (881 on this bundle),
so the prepared text is `pdf-inspector -> PP-OCRv5` with no Tesseract in the path.

Setup:
1. Build the upload locally: `python -m ondemand ocr-bundle --select targets`.
2. Add the resulting zip as a Kaggle **Dataset**, and attach it to this notebook.
3. Settings: **GPU T4 x2** (use one) and **Internet on** (pip needs it).
4. Seed the resume file with the 297 PP-OCR results you already have, so only ~584 pages are new.
5. Run top to bottom, then download `light_ocr_ppocrv5.zip` from the output.

Every loop resumes by unit, so re-running a cell after a crash only redoes what is missing.

In [ ]:
import glob
import json
import os
import subprocess
import time
import zipfile

WORK = "/kaggle/working/light_ocr"
os.makedirs(WORK, exist_ok=True)

bundle = next(iter(glob.glob("/kaggle/input/*/light_ocr_bundle*.zip")), None)
if bundle is None:
    raise SystemExit("attach the dataset holding light_ocr_bundle_targets.zip")
with zipfile.ZipFile(bundle) as zf:
    zf.extractall(WORK)
pages = [json.loads(line) for line in open(f"{WORK}/pages.jsonl")]
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(len(pages), "pages,", gpu)

## Optional: seed the resume file

Upload `light_ocr_ppocrv5.jsonl` from the earlier 297-page run as a second dataset. Those units are then skipped.

In [ ]:
seed = next(iter(glob.glob("/kaggle/input/*/light_ocr_ppocrv5.jsonl")), None)
if seed and not os.path.exists(f"{WORK}/light_ocr_ppocrv5.jsonl"):
    units = {p["unit"] for p in pages}
    kept = [l for l in open(seed) if json.loads(l)["unit"] in units]
    open(f"{WORK}/light_ocr_ppocrv5.jsonl", "w").writelines(kept)
    print("seeded", len(kept), "results")
else:
    print("no seed file; every page will be OCR'd")

## Render at 200 dpi

Same settings as the earlier run: 200 dpi, long side capped at 4000 px.

In [ ]:
!pip install -q pymupdf
import fitz

MAX_SIDE = 4000
os.makedirs(f"{WORK}/png", exist_ok=True)
for p in pages:
    p["png"] = f"{WORK}/png/" + os.path.basename(p["file"])[:-4] + ".png"
    if not os.path.exists(p["png"]):
        with fitz.open(f"{WORK}/" + p["file"]) as doc:
            page = doc[0]
            dpi = 200
            side = max(page.rect.width, page.rect.height) * dpi / 72
            if side > MAX_SIDE:
                dpi = int(dpi * MAX_SIDE / side)
            page.get_pixmap(dpi=dpi).save(p["png"])
print("rendered", len(pages), "pages")

## PP-OCRv5

Kaggle's T4 is CUDA 12.x, so the cu126 Paddle wheel applies. If the GPU is a P100 the wheel will not load;
switch the accelerator to T4 rather than falling back to CPU, which would take hours.

In [ ]:
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr>=3.0,<4"

In [ ]:
from paddleocr import PaddleOCR

started = time.perf_counter()
ocr = PaddleOCR(text_detection_model_name="PP-OCRv5_server_det", text_recognition_model_name="PP-OCRv5_server_rec",
                use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,
                device="gpu:0")
ocr.predict(pages[0]["png"])
load = time.perf_counter() - started
print(f"model ready in {load:.1f}s")

In [ ]:
def done_units(name):
    path = f"{WORK}/light_ocr_{name}.jsonl"
    return {json.loads(l)["unit"] for l in open(path)} if os.path.exists(path) else set()

def run_all(name, transcribe, meta):
    done = done_units(name)
    todo = [p for p in pages if p["unit"] not in done]
    print(f"{name}: {len(done)} done, {len(todo)} to go")
    with open(f"{WORK}/light_ocr_{name}.jsonl", "a") as f:
        for i, p in enumerate(todo, 1):
            started = time.perf_counter()
            try:
                text, error = transcribe(p), None
            except Exception as exc:
                text, error = "", f"{type(exc).__name__}: {exc}"[:200]
                print("fail", p["unit"], error)
            f.write(json.dumps({"unit": p["unit"], "text": text,
                                "seconds": time.perf_counter() - started, "error": error},
                               ensure_ascii=False) + "\n")
            f.flush()
            if i % 50 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)}")
    json.dump({**meta, "gpu": gpu}, open(f"{WORK}/meta_{name}.json", "w"))

def paddle_text(p):
    return "\n".join(t for r in ocr.predict(p["png"]) for t in r["rec_texts"])

run_all("ppocrv5", paddle_text, {"model": "PP-OCRv5_server_det + PP-OCRv5_server_rec", "load_seconds": load})

## Line boxes and confidences

`light_ocr.merge_pure` reads the confidences from this file, and the debugging portal re-renders the polygons.

In [ ]:
def paddle_lines(p):
    out = []
    for r in ocr.predict(p["png"]):
        polys = r.get("rec_polys", r.get("dt_polys", []))
        for text, score, poly in zip(r["rec_texts"], r["rec_scores"], polys):
            out.append({"text": text, "score": float(score), "poly": [[float(x), float(y)] for x, y in poly]})
    return out

path = f"{WORK}/light_ocr_boxes_ppocrv5.jsonl"
done = {json.loads(l)["unit"] for l in open(path)} if os.path.exists(path) else set()
todo = [p for p in pages if p["unit"] not in done]
print(f"boxes: {len(done)} done, {len(todo)} to go")
with open(path, "a") as f:
    for i, p in enumerate(todo, 1):
        try:
            lines, error = paddle_lines(p), None
        except Exception as exc:
            lines, error = [], f"{type(exc).__name__}: {exc}"[:200]
        f.write(json.dumps({"unit": p["unit"], "lines": lines, "error": error}, ensure_ascii=False) + "\n")
        f.flush()
        if i % 50 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)}")

## Timing and download

`seconds_per_page` here is a Kaggle T4 number. Keep the GPU name next to it in any sheet.

In [ ]:
import statistics

rows = [json.loads(l) for l in open(f"{WORK}/light_ocr_ppocrv5.jsonl")]
seconds = [r["seconds"] for r in rows]
print({"pages": len(rows), "failed": sum(bool(r["error"]) for r in rows),
       "empty": sum(not r["text"].strip() for r in rows),
       "total_seconds": round(sum(seconds), 1),
       "seconds_per_page": round(statistics.mean(seconds), 3), "gpu": gpu})

with zipfile.ZipFile("/kaggle/working/light_ocr_ppocrv5.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(WORK)):
        if name.endswith((".jsonl", ".json")) and name != "pages.jsonl":
            zf.write(f"{WORK}/{name}", name)
print("wrote /kaggle/working/light_ocr_ppocrv5.zip")